# Running SPICE Simulations

## CircuitBench Tutorial 3

This notebook demonstrates how CircuitBench interfaces with SPICE simulators to generate reproducible benchmark datasets for machine-learning surrogate modeling.

### Learning Objectives

- Configure a SPICE simulator
- Load circuit netlists
- Validate simulation inputs
- Configure parameter sweeps
- Execute simulations
- Store simulation results
- Convert simulations into machine-learning datasets


## Background

SPICE (Simulation Program with Integrated Circuit Emphasis) remains the industry standard for transistor-level circuit simulation. CircuitBench automates SPICE execution and converts simulation outputs into benchmark-ready datasets.

In [ ]:
import json
from pathlib import Path
import warnings
import random
import numpy as np

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
from circuitbench.simulation import SpiceRunner
from circuitbench.datasets import DatasetBuilder

## Initialize SPICE Runner

In [ ]:
runner = SpiceRunner()

## Available SPICE Engines

In [ ]:
runner.available_engines()

Typical supported simulators include:

- Ngspice
- Xyce
- LTspice
- HSPICE
- Spectre (if licensed)

## Select Simulator

In [ ]:
runner.set_engine("ngspice")

## Verify Installation

In [ ]:
runner.check_installation()

## Locate Circuit Netlists

In [ ]:
netlist_directory = Path("circuits")
sorted(netlist_directory.glob("*.cir"))

## Load a Netlist

In [ ]:
netlist = runner.load_netlist("circuits/CMOS_Inverter.cir")

netlist

## Validate Netlist

In [ ]:
validation = runner.validate_netlist(netlist)
validation

## Simulation Parameters

In [ ]:
simulation_config = {
    "analysis": "dc",
    "temperature": 27,
    "supply_voltage": 1.8,
    "process_corner": "typical",
}

simulation_config

## Parameter Sweep

In [ ]:
parameter_space = {
    "Vin": np.linspace(0.0, 1.8, 101),
    "Temperature": [0, 27, 75],
    "VDD": [1.6, 1.8, 2.0],
}

parameter_space

## Execute Parameter Sweep

In [ ]:
results = runner.parameter_sweep(
    netlist=netlist, parameters=parameter_space, config=simulation_config
)

## Simulation Summary

In [ ]:
results.summary()

## Preview Results

In [ ]:
results.dataframe.head()

## Save Raw Simulation Results

In [ ]:
output_dir = Path("simulation_results")
output_dir.mkdir(exist_ok=True)

results.dataframe.to_csv(output_dir / "raw_simulation_results.csv", index=False)

## Convert Simulations into a Machine Learning Dataset

In [ ]:
builder = DatasetBuilder()

In [ ]:
dataset = builder.from_simulation_results(results)

## Dataset Overview

In [ ]:
dataset.summary()

## Preview Machine Learning Dataset

In [ ]:
dataset.dataframe.head()

## Dataset Statistics

In [ ]:
dataset.dataframe.describe()

## Feature Correlation

In [ ]:
corr = dataset.dataframe.corr(numeric_only=True)
corr

## Correlation Heatmap

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 10))
plt.imshow(corr, interpolation="nearest", aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.colorbar(label="Correlation")
plt.title("Simulation Feature Correlation")
plt.tight_layout()

## Export Benchmark Dataset

In [ ]:
dataset.export(output_dir / "benchmark_dataset.csv")

## Export Dataset Metadata

In [ ]:
metadata = {
    "simulator": "ngspice",
    "analysis": simulation_config["analysis"],
    "temperature": simulation_config["temperature"],
    "process_corner": simulation_config["process_corner"],
    "random_seed": SEED,
    "samples": len(dataset.dataframe),
}
with open(output_dir / "simulation_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

## Reproducibility Checklist

- Fixed random seed
- SPICE engine recorded
- Simulation parameters documented
- Parameter sweep configuration saved
- Raw simulation data exported
- Benchmark dataset generated
- Metadata archived


## Best Practices

1. Always validate the netlist before simulation.
2. Record simulator version and process corner.
3. Keep simulation parameters under version control.
4. Export raw simulation results before preprocessing.
5. Use reproducible random seeds when generating benchmark datasets.


## Next Notebook

Continue with **4_Baseline_Models.ipynb** to train and compare baseline machine learning surrogate models using the datasets generated from SPICE simulations.

# Summary

In this notebook we:

- Configured a SPICE simulation environment.
- Loaded and validated a circuit netlist.
- Defined simulation parameters and parameter sweeps.
- Executed SPICE simulations.
- Converted simulation outputs into a benchmark-ready machine learning dataset.
- Exported simulation results and metadata.
- Established a reproducible workflow for surrogate model benchmarking.
